In [1]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle
import json


In [2]:
# Load trained model
model = load_model(r"Next_word_prediction_model.h5")

token_path = r"E:\1_Data_Science\Machine-Learning-Projects\7.Next_word_Prediction_Model_using_DL\Pickle File\tokenizer.pkl"

config_path = r"E:\1_Data_Science\Machine-Learning-Projects\7.Next_word_Prediction_Model_using_DL\Pickle File\config.json"

# Load tokenizer
with open(token_path, "rb") as f:
    tokenizer = pickle.load(f)

# Load config
with open(config_path, "r") as f:
    config = json.load(f)

max_sequence_len = config["max_sequence_len"]


In [3]:
# index → word mapping (FAST)
index_to_word = {index: word for word, index in tokenizer.word_index.items()}


In [4]:
def predict_next_words(
    seed_text: str,
    num_words: int = 3
) -> str:
    """
    Generate next words for a given seed text using a trained LSTM model.

    Args:
        seed_text (str): Initial input text
        num_words (int): Number of words to generate

    Returns:
        str: Text with generated words appended
    """

    if not seed_text or not isinstance(seed_text, str):
        raise ValueError("seed_text must be a non-empty string")

    for _ in range(num_words):

        # Convert text to sequence
        token_list = tokenizer.texts_to_sequences([seed_text])[0]

        # Handle unknown words
        if len(token_list) == 0:
            break

        # Pad sequence
        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_len - 1,
            padding="pre"
        )

        # Predict next word index
        predicted_index = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

        # Convert index to word
        next_word = index_to_word.get(predicted_index, "")

        if next_word == "":
            break

        # Append predicted word
        seed_text += " " + next_word

    return seed_text


In [12]:
#use it 
text = predict_next_words(
    seed_text = "i have ",
    num_words = 5
)

print(text)

i have  been beaten four times three
